# Source Profiling Framework

## Purpose

Automatically profiles every dataset in the OneLake landing zone.

## Outputs

- Table Statistics
- Column Statistics
- Null Analysis
- Duplicate Analysis
- Cardinality
- Data Analysis
- Relationship Checks

## Target

Lakehouse Profiling Folder

In [2]:
from pathlib import Path
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timezone
import json
import uuid
import time

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 4, Finished, Available, Finished, False)

In [3]:
LAKEHOUSE_ROOT = "/lakehouse/default/"
SOURCE_FOLDER = f"{LAKEHOUSE_ROOT}/Files/landing/olist/source_csv"
PROFILE_FOLDER = f"{LAKEHOUSE_ROOT}/Files/profiling"
REPORT_FOLDER = f"{PROFILE_FOLDER}/reports"
STATISTICS_FOLDER = f"{PROFILE_FOLDER}/statistics"
LOG_FOLDER = f"{PROFILE_FOLDER}/logs"

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 5, Finished, Available, Finished, False)

# Profiling Run Metadata

In [4]:
PROFILE_RUN_ID = str(uuid.uuid4())
PROFILE_START_TIME = datetime.now(timezone.utc)
PROFILE_START_PERF = time.perf_counter()

print("Profiling run ID: ", PROFILE_RUN_ID)
print("Profiling started at: ", PROFILE_START_TIME)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 6, Finished, Available, Finished, False)

Profiling run ID:  3f45bfdb-6dd0-4e53-9b54-8af209cff497
Profiling started at:  2026-07-28 06:36:18.503976+00:00


In [5]:
import os

for folder in [PROFILE_FOLDER, REPORT_FOLDER, STATISTICS_FOLDER, LOG_FOLDER]:
    os.makedirs(folder, exist_ok=True)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 7, Finished, Available, Finished, False)

In [6]:
csv_files = list(Path(SOURCE_FOLDER).glob("*.csv"))

print(len(csv_files))
print(csv_files)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 8, Finished, Available, Finished, False)

9
[PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_customers_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_geolocation_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_items_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_payments_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_reviews_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_orders_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_products_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_sellers_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/product_category_name_translation.csv')]


In [7]:
from pathlib import Path

datasets = {}

for file in csv_files:
    dataset_name = file.stem

    spark_path = (
        f"Files/landing/olist/source_csv/{file.name}"
    )

    datasets[dataset_name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(spark_path)
    )

    print(f"Loaded: {dataset_name}")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 9, Finished, Available, Finished, False)

Loaded: olist_customers_dataset
Loaded: olist_geolocation_dataset
Loaded: olist_order_items_dataset
Loaded: olist_order_payments_dataset
Loaded: olist_order_reviews_dataset
Loaded: olist_orders_dataset
Loaded: olist_products_dataset
Loaded: olist_sellers_dataset
Loaded: product_category_name_translation


# Verifying if Datasets loaded correctly

In [8]:
for dataset_name,df in datasets.items():
    print("="*20)
    print(f"Dataset: {dataset_name}")
    print(f"Rows: {df.count():,}")
    print(f"Columns: {len(df.columns)}")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 10, Finished, Available, Finished, False)

Dataset: olist_customers_dataset
Rows: 99,441
Columns: 5
Dataset: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Dataset: olist_order_items_dataset
Rows: 112,650
Columns: 7
Dataset: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Dataset: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Dataset: olist_orders_dataset
Rows: 99,441
Columns: 8
Dataset: olist_products_dataset
Rows: 32,951
Columns: 9
Dataset: olist_sellers_dataset
Rows: 3,095
Columns: 4
Dataset: product_category_name_translation
Rows: 71
Columns: 2


# Generating table level profiling statistical summary

In [9]:
table_summary = []

for dataset_name, df in datasets.items():
    row_count = df.count()
    column_count = len(df.columns)

    table_summary.append({
        "table_name": dataset_name,
        "row_count": row_count,
        "column_count": column_count,
        "profiled_at": datetime.now()
    })

table_summary_df = pd.DataFrame(table_summary)

display(table_summary_df)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28bf8be5-a24e-4bb0-9779-47eb727859f8)

In [10]:
spark.createDataFrame(table_summary_df).write.mode("overwrite").format("delta").saveAsTable('profile_table_summary')

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 12, Finished, Available, Finished, False)

# Column Level Profiling

In [11]:
import builtins

column_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    for field in df.schema.fields:

        column_name = str(field.name)
        data_type = str(field.dataType)
        nullable = field.nullable

        escaped_name = column_name.replace("`", "``")
        column_ref = F.col(f"`{escaped_name}`")

        null_count = (
            df.filter(column_ref.isNull())
            .count()
        )

        distinct_count = (
            df.select(column_ref)
            .distinct()
            .count()
        )

        null_percentage = (
            builtins.round(
                (null_count / total_rows) * 100,
                2
            )
            if total_rows > 0
            else 0.0
        )

        column_summary.append({
            "table_name": dataset_name,
            "column_name": column_name,
            "data_type": data_type,
            "nullable": nullable,
            "row_count": total_rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "distinct_values": distinct_count
        })

print("Column summary records created:", len(column_summary))

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 13, Finished, Available, Finished, False)

Column summary records created: 52


In [12]:
column_summary_df = pd.DataFrame(column_summary)
display(column_summary_df.head(20))

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69bc8756-61f3-4f85-824f-33d8e0caba8b)

In [13]:
spark.createDataFrame(column_summary_df).write.mode("overwrite").format("delta").saveAsTable("profile_column_summary")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 15, Finished, Available, Finished, False)

# Null Analysis

In [14]:
null_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    for field in df.schema.fields:

        column_name = str(field.name)

        # Escape column names safely
        escaped_name = column_name.replace("`", "``")
        column_ref = F.col(f"`{escaped_name}`")

        null_count = (
            df.filter(column_ref.isNull())
            .count()
        )

        null_percentage = (
            builtins.round(
                (null_count / total_rows) * 100,
                2
            )
            if total_rows > 0
            else 0.0
        )

        not_null_count = total_rows - null_count

        not_null_percentage = builtins.round(
            100.0 - null_percentage,
            2
        )

        null_summary.append({
            "table_name": dataset_name,
            "column_name": column_name,
            "row_count": total_rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "not_null_count": not_null_count,
            "not_null_percentage": not_null_percentage
        })

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 16, Finished, Available, Finished, False)

In [15]:
null_summary_df = pd.DataFrame(null_summary)
display(
    null_summary_df.sort_values(
        "null_percentage",
        ascending=False
    )
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0eed6fce-1b58-4d22-b56c-67a504b1ac15)

In [16]:
spark.createDataFrame(null_summary_df)\
    .write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("profile_null_summary")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 18, Finished, Available, Finished, False)

# Duplicate and Key Analysis

In [17]:
BUSINESS_KEYS = {
    "olist_customers_dataset": ["customer_id"],
    "olist_geolocation_dataset": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": [
        "order_id",
        "payment_sequential"
    ],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": [
        "product_category_name"
    ]
}

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 19, Finished, Available, Finished, False)

In [18]:
duplicate_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    # Exact duplicate rows
    distinct_rows = df.distinct().count()
    exact_duplicate_rows = total_rows - distinct_rows

    key_columns = BUSINESS_KEYS.get(dataset_name, [])

    missing_key_columns = [
        column_name
        for column_name in key_columns
        if column_name not in df.columns
    ]

    if missing_key_columns:
        raise ValueError(
            f"{dataset_name} is missing key columns: "
            f"{missing_key_columns}"
        )

    if key_columns:
        duplicate_key_groups = (
            df.groupBy(*key_columns)
              .count()
              .filter(F.col("count") > 1)
        )

        duplicate_key_group_count = (
            duplicate_key_groups.count()
        )

        duplicate_key_row_count = (
            duplicate_key_groups
            .select(
                F.sum(
                    F.col("count") - F.lit(1)
                ).alias("duplicate_rows")
            )
            .collect()[0]["duplicate_rows"]
        )

        duplicate_key_row_count = (
            int(duplicate_key_row_count)
            if duplicate_key_row_count is not None
            else 0
        )

        distinct_key_count = (
            df.select(*key_columns)
              .distinct()
              .count()
        )

    else:
        duplicate_key_group_count = None
        duplicate_key_row_count = None
        distinct_key_count = None

    exact_duplicate_percentage = (
        builtins.round(
            (exact_duplicate_rows / total_rows) * 100,
            4
        )
        if total_rows > 0
        else 0.0
    )

    duplicate_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "table_name": dataset_name,
        "business_key": ", ".join(key_columns),
        "row_count": total_rows,
        "distinct_row_count": distinct_rows,
        "exact_duplicate_row_count": exact_duplicate_rows,
        "exact_duplicate_percentage": exact_duplicate_percentage,
        "distinct_business_key_count": distinct_key_count,
        "duplicate_key_group_count": duplicate_key_group_count,
        "duplicate_business_key_row_count": duplicate_key_row_count
    })

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 20, Finished, Available, Finished, False)

In [19]:
duplicate_summary_spark_df = spark.createDataFrame(duplicate_summary)

display(
    duplicate_summary_spark_df
    .orderBy(
        F.col("exact_duplicate_row_count").desc(),
        F.col("duplicate_business_key_row_count").desc_nulls_last()
    )
)

(
    duplicate_summary_spark_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_duplicate_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69baace8-34f2-4dbe-ab30-40d753d32e9b)

In [20]:
def show_duplicate_keys(dataset_name: str, limit: int = 20):
    df = datasets[dataset_name]
    key_columns = BUSINESS_KEYS[dataset_name]

    duplicate_keys = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
    )

    display(duplicate_keys.limit(limit))

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 22, Finished, Available, Finished, False)

In [21]:
show_duplicate_keys("olist_geolocation_dataset")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bf4f94c-3739-4c2e-a490-64a3f1b4869f)

# Business-key Completeness

Measure null values in expected source business-key columns.

In [22]:
key_quality_summary = []

for dataset_name, key_columns in BUSINESS_KEYS.items():

    df = datasets[dataset_name]
    total_rows = df.count()

    null_condition = None

    for column_name in key_columns:

        column_ref = F.col(column_name)

        condition = (
            column_ref.isNull()
            | (F.trim(column_ref) == "")
        )

        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    rows_with_missing_key = (
        df.filter(null_condition)
        .count()
    )

    missing_key_percentage = (
        builtins.round(
            (rows_with_missing_key / total_rows) * 100,
            4
        )
        if total_rows > 0
        else 0.0
    )

    key_quality_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "table_name": dataset_name,
        "business_key": ", ".join(key_columns),
        "row_count": total_rows,
        "rows_with_missing_key": rows_with_missing_key,
        "missing_key_percentage": missing_key_percentage
    })

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 24, Finished, Available, Finished, False)

In [23]:
key_quality_spark_df = spark.createDataFrame(key_quality_summary)

display(
    key_quality_spark_df
    .orderBy(
        F.col("rows_with_missing_key").desc()
    )
)

(
    key_quality_spark_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_key_quality_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df57cc08-acec-40f9-9a8c-943c3b8ed205)

In [24]:
PROFILE_END_TIME = datetime.now(timezone.utc)

PROFILE_DURATION_SECONDS = builtins.round(
    time.perf_counter() - PROFILE_START_PERF,
    2
)

TOTAL_TABLES = len(datasets)

TOTAL_COLUMNS = builtins.sum(
    len(df.columns)
    for df in datasets.values()
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 26, Finished, Available, Finished, False)

In [25]:
profiling_run_record = [{
    "profile_run_id": PROFILE_RUN_ID,
    "started_at_utc": PROFILE_START_TIME,
    "completed_at_utc": PROFILE_END_TIME,
    "duration_seconds": PROFILE_DURATION_SECONDS,
    "dataset_count": len(csv_files),
    "table_count": TOTAL_TABLES,
    "column_count": TOTAL_COLUMNS,
    "status": "SUCCESS"
}]

profiling_run_df = spark.createDataFrame(profiling_run_record)

display(profiling_run_df)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 550a0854-cf87-472c-8817-d92a12b84135)

In [26]:
(
    profiling_run_df
    .write
    .mode("append")
    .format("delta")
    .saveAsTable("profile_run_history")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 28, Finished, Available, Finished, False)

# Numeric Profiling

In [28]:
NUMERIC_TYPES = (
    "IntegerType",
    "LongType",
    "FloatType",
    "DoubleType",
    "DecimalType",
    "ShortType"
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 31, Finished, Available, Finished, False)

In [34]:
NUMERIC_COLUMN_CONFIG = {
    "olist_customers_dataset": [
        "customer_zip_code_prefix"
    ],

    "olist_geolocation_dataset": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ],

    "olist_order_items_dataset": [
        "order_item_id",
        "price",
        "freight_value"
    ],

    "olist_order_payments_dataset": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],

    "olist_order_reviews_dataset": [
        "review_score"
    ],

    "olist_products_dataset": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],

    "olist_sellers_dataset": [
        "seller_zip_code_prefix"
    ]
}

numeric_summary = []

for dataset_name, numeric_columns in NUMERIC_COLUMN_CONFIG.items():

    df = datasets[dataset_name]
    total_rows = df.count()

    for column_name in numeric_columns:

        if column_name not in df.columns:
            raise ValueError(
                f"Column '{column_name}' was not found "
                f"in dataset '{dataset_name}'."
            )

        source_column = F.trim(F.col(column_name))

        numeric_column = source_column.cast("double")

        stats = (
            df.select(
                source_column.alias("source_value"),
                numeric_column.alias("numeric_value")
            )
            .agg(
                F.count("*").alias("row_count"),

                F.sum(
                    F.when(
                        F.col("source_value").isNull()
                        | (F.col("source_value") == ""),
                        1
                    ).otherwise(0)
                ).alias("null_or_blank_count"),

                F.sum(
                    F.when(
                        F.col("source_value").isNotNull()
                        & (F.col("source_value") != "")
                        & F.col("numeric_value").isNull(),
                        1
                    ).otherwise(0)
                ).alias("invalid_numeric_count"),

                F.sum(
                    F.when(
                        F.col("numeric_value").isNotNull(),
                        1
                    ).otherwise(0)
                ).alias("valid_numeric_count"),

                F.min("numeric_value").alias("minimum"),

                F.max("numeric_value").alias("maximum"),

                F.avg("numeric_value").alias("average"),

                F.expr(
                    "percentile_approx(numeric_value, 0.5)"
                ).alias("median"),

                F.stddev("numeric_value").alias(
                    "standard_deviation"
                )
            )
            .collect()[0]
        )

        valid_numeric_count = int(
            stats["valid_numeric_count"] or 0
        )

        invalid_numeric_count = int(
            stats["invalid_numeric_count"] or 0
        )

        null_or_blank_count = int(
            stats["null_or_blank_count"] or 0
        )

        numeric_summary.append({
            "profile_run_id": PROFILE_RUN_ID,
            "table_name": dataset_name,
            "column_name": column_name,
            "source_data_type": "string",
            "target_numeric_type": "double",
            "row_count": total_rows,
            "valid_numeric_count": valid_numeric_count,
            "invalid_numeric_count": invalid_numeric_count,
            "null_or_blank_count": null_or_blank_count,
            "valid_numeric_percentage": builtins.round(
                valid_numeric_count / total_rows * 100,
                4
            ) if total_rows > 0 else 0.0,
            "invalid_numeric_percentage": builtins.round(
                invalid_numeric_count / total_rows * 100,
                4
            ) if total_rows > 0 else 0.0,
            "minimum": (
                float(stats["minimum"])
                if stats["minimum"] is not None
                else None
            ),
            "maximum": (
                float(stats["maximum"])
                if stats["maximum"] is not None
                else None
            ),
            "average": (
                float(stats["average"])
                if stats["average"] is not None
                else None
            ),
            "median": (
                float(stats["median"])
                if stats["median"] is not None
                else None
            ),
            "standard_deviation": (
                float(stats["standard_deviation"])
                if stats["standard_deviation"] is not None
                else None
            )
        })



StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 40, Finished, Available, Finished, False)

In [37]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

numeric_profile_schema = StructType([
    StructField(
        "profile_run_id",
        StringType(),
        False
    ),
    StructField(
        "table_name",
        StringType(),
        False
    ),
    StructField(
        "column_name",
        StringType(),
        False
    ),
    StructField(
        "source_data_type",
        StringType(),
        False
    ),
    StructField(
        "target_numeric_type",
        StringType(),
        False
    ),
    StructField(
        "row_count",
        LongType(),
        False
    ),
    StructField(
        "valid_numeric_count",
        LongType(),
        False
    ),
    StructField(
        "invalid_numeric_count",
        LongType(),
        False
    ),
    StructField(
        "null_or_blank_count",
        LongType(),
        False
    ),
    StructField(
        "valid_numeric_percentage",
        DoubleType(),
        False
    ),
    StructField(
        "invalid_numeric_percentage",
        DoubleType(),
        False
    ),
    StructField(
        "minimum",
        DoubleType(),
        True
    ),
    StructField(
        "maximum",
        DoubleType(),
        True
    ),
    StructField(
        "average",
        DoubleType(),
        True
    ),
    StructField(
        "median",
        DoubleType(),
        True
    ),
    StructField(
        "standard_deviation",
        DoubleType(),
        True
    )
])

if not numeric_summary:
    raise ValueError(
        "No numeric profiling records were generated. "
        "Check NUMERIC_COLUMN_CONFIG and source columns."
    )

numeric_summary_df = spark.createDataFrame(
    numeric_summary,
    schema=numeric_profile_schema
)

display(
    numeric_summary_df.orderBy(
        "table_name",
        "column_name"
    )
)

(
    numeric_summary_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_numeric_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 93b5dcff-9769-4924-bdb9-14b79c5338f1)

# Datetime Profiling

Generate statistics for timestamp and date columns.

In [38]:
DATE_COLUMNS = [
    "timestamp",
    "date"
]

datetime_summary = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        lower = column.lower()

        if not any(keyword in lower for keyword in DATE_COLUMNS):
            continue
        
        parsed = F.to_timestamp(column)

        summary = (
            df.select(
                F.min(parsed).alias("minimum"),
                F.max(parsed).alias("maximum"),
                F.sum(
                    F.when(
                        parsed.isNull(),
                        1
                    ).otherwise(0)
                ).alias("invalid_dates")
            )
            .collect()[0]
        )

        datetime_summary.append({
            "profile_run_id": PROFILE_RUN_ID,
            "table_name": dataset_name,
            "column_name": column,
            "minimum_date": summary["minimum"],
            "maximum_date": summary["maximum"],
            "invalid_dates": summary["invalid_dates"]
        })

spark.createDataFrame(datetime_summary).write.mode("overwrite").saveAsTable("profile_datetime_summary")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 44, Finished, Available, Finished, False)

# Categorical Profiling

In [42]:
CATEGORICAL_COLUMN_CONFIG = {
    "olist_customers_dataset": [
        "customer_city",
        "customer_state"
    ],

    "olist_orders_dataset": [
        "order_status"
    ],

    "olist_order_payments_dataset": [
        "payment_type"
    ],

    "olist_products_dataset": [
        "product_category_name"
    ],

    "olist_sellers_dataset": [
        "seller_city",
        "seller_state"
    ],

    "olist_geolocation_dataset": [
        "geolocation_city",
        "geolocation_state"
    ],

    "product_category_name_translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}

categorical_summary = []

for dataset_name, categorical_columns in CATEGORICAL_COLUMN_CONFIG.items():

    df = datasets[dataset_name]
    total_rows = df.count()

    for column_name in categorical_columns:

        if column_name not in df.columns:
            raise ValueError(
                f"Column '{column_name}' was not found "
                f"in dataset '{dataset_name}'."
            )

        cleaned_column = F.when(
            F.trim(F.col(column_name)) == "",
            None
        ).otherwise(
            F.trim(F.col(column_name))
        )

        prepared_df = df.select(
            cleaned_column.alias("category_value")
        )

        summary_stats = (
            prepared_df
            .agg(
                F.count("*").alias("row_count"),

                F.sum(
                    F.when(
                        F.col("category_value").isNull(),
                        1
                    ).otherwise(0)
                ).alias("null_or_blank_count"),

                F.countDistinct(
                    "category_value"
                ).alias("distinct_non_null_values")
            )
            .collect()[0]
        )

        top_row = (
            prepared_df
            .filter(
                F.col("category_value").isNotNull()
            )
            .groupBy("category_value")
            .count()
            .orderBy(
                F.desc("count"),
                F.asc("category_value")
            )
            .first()
        )

        distinct_non_null_values = int(
            summary_stats["distinct_non_null_values"] or 0
        )

        null_or_blank_count = int(
            summary_stats["null_or_blank_count"] or 0
        )

        non_null_count = total_rows - null_or_blank_count

        top_value = (
            top_row["category_value"]
            if top_row is not None
            else None
        )

        top_value_count = (
            int(top_row["count"])
            if top_row is not None
            else 0
        )

        categorical_summary.append({
            "profile_run_id": PROFILE_RUN_ID,
            "table_name": dataset_name,
            "column_name": column_name,
            "row_count": total_rows,
            "null_or_blank_count": null_or_blank_count,
            "non_null_count": non_null_count,
            "distinct_non_null_values": distinct_non_null_values,
            "distinct_percentage": (
                builtins.round(
                    distinct_non_null_values / non_null_count * 100,
                    4
                )
                if non_null_count > 0
                else 0.0
            ),
            "top_value": top_value,
            "top_value_count": top_value_count,
            "top_value_percentage": (
                builtins.round(
                    top_value_count / non_null_count * 100,
                    4
                )
                if non_null_count > 0
                else 0.0
            )
        })

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

categorical_profile_schema = StructType([
    StructField(
        "profile_run_id",
        StringType(),
        False
    ),
    StructField(
        "table_name",
        StringType(),
        False
    ),
    StructField(
        "column_name",
        StringType(),
        False
    ),
    StructField(
        "row_count",
        LongType(),
        False
    ),
    StructField(
        "null_or_blank_count",
        LongType(),
        False
    ),
    StructField(
        "non_null_count",
        LongType(),
        False
    ),
    StructField(
        "distinct_non_null_values",
        LongType(),
        False
    ),
    StructField(
        "distinct_percentage",
        DoubleType(),
        False
    ),
    StructField(
        "top_value",
        StringType(),
        True
    ),
    StructField(
        "top_value_count",
        LongType(),
        False
    ),
    StructField(
        "top_value_percentage",
        DoubleType(),
        False
    )
])

if not categorical_summary:
    raise ValueError(
        "No categorical profiling records were generated. "
        "Check CATEGORICAL_COLUMN_CONFIG."
    )

categorical_summary_df = spark.createDataFrame(
    categorical_summary,
    schema=categorical_profile_schema
)

display(
    categorical_summary_df.orderBy(
        "table_name",
        "column_name"
    )
)

(
    categorical_summary_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_categorical_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d91d9a5-2f03-4da6-9a39-b88d600ee120)

# Business Rule Validation

Validate source values against expected business conditions and store rule-level results for data-quality reporting.

In [43]:
VALID_ORDER_STATUSES = [
    "approved",
    "canceled",
    "created",
    "delivered",
    "invoiced",
    "processing",
    "shipped",
    "unavailable"
]

business_rule_summary = []


def add_business_rule_result(
    rule_id,
    rule_name,
    dataset_name,
    column_name,
    rule_description,
    total_rows,
    evaluated_rows,
    passed_rows,
    failed_rows,
    null_or_blank_rows,
    severity="ERROR"
):
    failure_percentage = (
        builtins.round(failed_rows / evaluated_rows * 100, 4)
        if evaluated_rows > 0
        else 0.0
    )

    pass_percentage = (
        builtins.round(passed_rows / evaluated_rows * 100, 4)
        if evaluated_rows > 0
        else 0.0
    )

    business_rule_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "table_name": dataset_name,
        "column_name": column_name,
        "rule_description": rule_description,
        "severity": severity,
        "row_count": int(total_rows),
        "evaluated_row_count": int(evaluated_rows),
        "passed_row_count": int(passed_rows),
        "failed_row_count": int(failed_rows),
        "null_or_blank_row_count": int(null_or_blank_rows),
        "pass_percentage": float(pass_percentage),
        "failure_percentage": float(failure_percentage),
        "status": "PASS" if failed_rows == 0 else "FAIL"
    })

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 49, Finished, Available, Finished, False)

In [44]:
reviews_df = datasets["olist_order_reviews_dataset"]

review_score_source = F.trim(F.col("review_score"))
review_score_numeric = review_score_source.cast("int")

review_score_stats = (
    reviews_df
    .select(
        review_score_source.alias("source_value"),
        review_score_numeric.alias("numeric_value")
    )
    .agg(
        F.count("*").alias("row_count"),

        F.sum(
            F.when(
                F.col("source_value").isNull()
                | (F.col("source_value") == ""),
                1
            ).otherwise(0)
        ).alias("null_or_blank_rows"),

        F.sum(
            F.when(
                F.col("numeric_value").between(1, 5),
                1
            ).otherwise(0)
        ).alias("passed_rows"),

        F.sum(
            F.when(
                F.col("source_value").isNotNull()
                & (F.col("source_value") != "")
                & (
                    F.col("numeric_value").isNull()
                    | ~F.col("numeric_value").between(1, 5)
                ),
                1
            ).otherwise(0)
        ).alias("failed_rows")
    )
    .collect()[0]
)

evaluated_rows = (
    review_score_stats["row_count"]
    - review_score_stats["null_or_blank_rows"]
)

add_business_rule_result(
    rule_id="BR001",
    rule_name="Valid review score",
    dataset_name="olist_order_reviews_dataset",
    column_name="review_score",
    rule_description="Review score must be an integer between 1 and 5.",
    total_rows=review_score_stats["row_count"],
    evaluated_rows=evaluated_rows,
    passed_rows=review_score_stats["passed_rows"],
    failed_rows=review_score_stats["failed_rows"],
    null_or_blank_rows=review_score_stats["null_or_blank_rows"],
    severity="ERROR"
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 50, Finished, Available, Finished, False)

In [45]:
def evaluate_non_negative_rule(
    rule_id,
    rule_name,
    dataset_name,
    column_name,
    rule_description,
    severity="ERROR"
):
    df = datasets[dataset_name]

    source_value = F.trim(F.col(column_name))
    numeric_value = source_value.cast("double")

    stats = (
        df
        .select(
            source_value.alias("source_value"),
            numeric_value.alias("numeric_value")
        )
        .agg(
            F.count("*").alias("row_count"),

            F.sum(
                F.when(
                    F.col("source_value").isNull()
                    | (F.col("source_value") == ""),
                    1
                ).otherwise(0)
            ).alias("null_or_blank_rows"),

            F.sum(
                F.when(
                    F.col("numeric_value") >= 0,
                    1
                ).otherwise(0)
            ).alias("passed_rows"),

            F.sum(
                F.when(
                    F.col("source_value").isNotNull()
                    & (F.col("source_value") != "")
                    & (
                        F.col("numeric_value").isNull()
                        | (F.col("numeric_value") < 0)
                    ),
                    1
                ).otherwise(0)
            ).alias("failed_rows")
        )
        .collect()[0]
    )

    evaluated_rows = (
        stats["row_count"]
        - stats["null_or_blank_rows"]
    )

    add_business_rule_result(
        rule_id=rule_id,
        rule_name=rule_name,
        dataset_name=dataset_name,
        column_name=column_name,
        rule_description=rule_description,
        total_rows=stats["row_count"],
        evaluated_rows=evaluated_rows,
        passed_rows=stats["passed_rows"],
        failed_rows=stats["failed_rows"],
        null_or_blank_rows=stats["null_or_blank_rows"],
        severity=severity
    )

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 51, Finished, Available, Finished, False)

In [46]:
evaluate_non_negative_rule(
    rule_id="BR002",
    rule_name="Non-negative item price",
    dataset_name="olist_order_items_dataset",
    column_name="price",
    rule_description="Item price must be numeric and greater than or equal to zero."
)

evaluate_non_negative_rule(
    rule_id="BR003",
    rule_name="Non-negative freight value",
    dataset_name="olist_order_items_dataset",
    column_name="freight_value",
    rule_description="Freight value must be numeric and greater than or equal to zero."
)

evaluate_non_negative_rule(
    rule_id="BR004",
    rule_name="Non-negative payment value",
    dataset_name="olist_order_payments_dataset",
    column_name="payment_value",
    rule_description="Payment value must be numeric and greater than or equal to zero."
)

evaluate_non_negative_rule(
    rule_id="BR005",
    rule_name="Non-negative payment instalments",
    dataset_name="olist_order_payments_dataset",
    column_name="payment_installments",
    rule_description="Payment instalments must be numeric and greater than or equal to zero."
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 52, Finished, Available, Finished, False)

In [47]:
orders_df = datasets["olist_orders_dataset"]

status_value = F.lower(
    F.trim(F.col("order_status"))
)

order_status_stats = (
    orders_df
    .select(
        status_value.alias("status_value")
    )
    .agg(
        F.count("*").alias("row_count"),

        F.sum(
            F.when(
                F.col("status_value").isNull()
                | (F.col("status_value") == ""),
                1
            ).otherwise(0)
        ).alias("null_or_blank_rows"),

        F.sum(
            F.when(
                F.col("status_value").isin(VALID_ORDER_STATUSES),
                1
            ).otherwise(0)
        ).alias("passed_rows"),

        F.sum(
            F.when(
                F.col("status_value").isNotNull()
                & (F.col("status_value") != "")
                & ~F.col("status_value").isin(VALID_ORDER_STATUSES),
                1
            ).otherwise(0)
        ).alias("failed_rows")
    )
    .collect()[0]
)

evaluated_rows = (
    order_status_stats["row_count"]
    - order_status_stats["null_or_blank_rows"]
)

add_business_rule_result(
    rule_id="BR006",
    rule_name="Valid order status",
    dataset_name="olist_orders_dataset",
    column_name="order_status",
    rule_description=(
        "Order status must belong to the expected Olist status domain."
    ),
    total_rows=order_status_stats["row_count"],
    evaluated_rows=evaluated_rows,
    passed_rows=order_status_stats["passed_rows"],
    failed_rows=order_status_stats["failed_rows"],
    null_or_blank_rows=order_status_stats["null_or_blank_rows"],
    severity="ERROR"
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 53, Finished, Available, Finished, False)

In [50]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

business_rule_schema = StructType([
    StructField("profile_run_id", StringType(), False),
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("rule_description", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("evaluated_row_count", LongType(), False),
    StructField("passed_row_count", LongType(), False),
    StructField("failed_row_count", LongType(), False),
    StructField("null_or_blank_row_count", LongType(), False),
    StructField("pass_percentage", DoubleType(), False),
    StructField("failure_percentage", DoubleType(), False),
    StructField("status", StringType(), False)
])

if not business_rule_summary:
    raise ValueError(
        "No business-rule results were generated."
    )

business_rule_summary_df = spark.createDataFrame(
    business_rule_summary,
    schema=business_rule_schema
)

display(
    business_rule_summary_df.orderBy(
        "rule_id"
    )
)

(
    business_rule_summary_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_business_rule_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6cdb2cf-3cb3-4a02-bce3-45a93398e596)

# Referential Integrity Profiling

Validate expected parent-child relationships and identify orphan records before transformation.

In [54]:
RELATIONSHIP_CONFIG = [
    {
        "relationship_id": "REL001",
        "relationship_name": "Orders to Customers",
        "child_table": "olist_orders_dataset",
        "child_columns": ["customer_id"],
        "parent_table": "olist_customers_dataset",
        "parent_columns": ["customer_id"],
        "severity": "ERROR"
    },
    {
        "relationship_id": "REL002",
        "relationship_name": "Order Items to Orders",
        "child_table": "olist_order_items_dataset",
        "child_columns": ["order_id"],
        "parent_table": "olist_orders_dataset",
        "parent_columns": ["order_id"],
        "severity": "ERROR"
    },
    {
        "relationship_id": "REL003",
        "relationship_name": "Order Items to Products",
        "child_table": "olist_order_items_dataset",
        "child_columns": ["product_id"],
        "parent_table": "olist_products_dataset",
        "parent_columns": ["product_id"],
        "severity": "ERROR"
    },
    {
        "relationship_id": "REL004",
        "relationship_name": "Order Items to Sellers",
        "child_table": "olist_order_items_dataset",
        "child_columns": ["seller_id"],
        "parent_table": "olist_sellers_dataset",
        "parent_columns": ["seller_id"],
        "severity": "ERROR"
    },
    {
        "relationship_id": "REL005",
        "relationship_name": "Payments to Orders",
        "child_table": "olist_order_payments_dataset",
        "child_columns": ["order_id"],
        "parent_table": "olist_orders_dataset",
        "parent_columns": ["order_id"],
        "severity": "ERROR"
    },
    {
        "relationship_id": "REL006",
        "relationship_name": "Reviews to Orders",
        "child_table": "olist_order_reviews_dataset",
        "child_columns": ["order_id"],
        "parent_table": "olist_orders_dataset",
        "parent_columns": ["order_id"],
        "severity": "WARNING"
    }
]

for relationship in RELATIONSHIP_CONFIG:

    child_table = relationship["child_table"]
    parent_table = relationship["parent_table"]

    child_columns = relationship["child_columns"]
    parent_columns = relationship["parent_columns"]

    if child_table not in datasets:
        raise KeyError(
            f"Child dataset '{child_table}' was not loaded."
        )

    if parent_table not in datasets:
        raise KeyError(
            f"Parent dataset '{parent_table}' was not loaded."
        )

    child_missing_columns = [
        column
        for column in child_columns
        if column not in datasets[child_table].columns
    ]

    parent_missing_columns = [
        column
        for column in parent_columns
        if column not in datasets[parent_table].columns
    ]

    if child_missing_columns:
        raise ValueError(
            f"{child_table} is missing columns: "
            f"{child_missing_columns}"
        )

    if parent_missing_columns:
        raise ValueError(
            f"{parent_table} is missing columns: "
            f"{parent_missing_columns}"
        )

print(
    f"Validated {len(RELATIONSHIP_CONFIG)} "
    "relationship definitions."
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 60, Finished, Available, Finished, False)

Validated 6 relationship definitions.


In [56]:
relationship_summary = []
relationship_orphan_samples = {}

for relationship in RELATIONSHIP_CONFIG:

    relationship_id = relationship["relationship_id"]
    relationship_name = relationship["relationship_name"]

    child_table = relationship["child_table"]
    parent_table = relationship["parent_table"]

    child_columns = relationship["child_columns"]
    parent_columns = relationship["parent_columns"]

    severity = relationship["severity"]

    child_df = datasets[child_table]
    parent_df = datasets[parent_table]

    child_row_count = child_df.count()

    # Standardize child key names for the relationship test.
    child_key_df = child_df.select(
        *[
            F.trim(F.col(child_column))
             .alias(f"key_{index}")
            for index, child_column
            in enumerate(child_columns)
        ]
    )

    # Standardize parent key names so both sides have matching names.
    parent_key_df = (
        parent_df
        .select(
            *[
                F.trim(F.col(parent_column))
                 .alias(f"key_{index}")
                for index, parent_column
                in enumerate(parent_columns)
            ]
        )
        .dropDuplicates()
    )

    key_aliases = [
        f"key_{index}"
        for index in range(len(child_columns))
    ]

    missing_key_condition = None

    for key_alias in key_aliases:

        current_condition = (
            F.col(key_alias).isNull()
            | (F.col(key_alias) == "")
        )

        missing_key_condition = (
            current_condition
            if missing_key_condition is None
            else missing_key_condition | current_condition
        )

    child_missing_key_count = (
        child_key_df
        .filter(missing_key_condition)
        .count()
    )

    valid_child_keys_df = (
        child_key_df
        .filter(~missing_key_condition)
    )

    evaluated_child_row_count = valid_child_keys_df.count()

    orphan_rows_df = (
        valid_child_keys_df
        .join(
            parent_key_df,
            on=key_aliases,
            how="left_anti"
        )
    )

    orphan_row_count = orphan_rows_df.count()

    matched_row_count = (
        evaluated_child_row_count - orphan_row_count
    )

    orphan_percentage = (
        builtins.round(
            orphan_row_count
            / evaluated_child_row_count
            * 100,
            4
        )
        if evaluated_child_row_count > 0
        else 0.0
    )

    relationship_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "relationship_id": relationship_id,
        "relationship_name": relationship_name,
        "child_table": child_table,
        "child_columns": ", ".join(child_columns),
        "parent_table": parent_table,
        "parent_columns": ", ".join(parent_columns),
        "severity": severity,
        "child_row_count": int(child_row_count),
        "evaluated_child_row_count": int(
            evaluated_child_row_count
        ),
        "child_missing_key_count": int(
            child_missing_key_count
        ),
        "matched_row_count": int(matched_row_count),
        "orphan_row_count": int(orphan_row_count),
        "orphan_percentage": float(orphan_percentage),
        "status": (
            "PASS"
            if orphan_row_count == 0
            else "FAIL"
        )
    })

    relationship_orphan_samples[
        relationship_id
    ] = orphan_rows_df.limit(20)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 62, Finished, Available, Finished, False)

In [57]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

relationship_profile_schema = StructType([
    StructField(
        "profile_run_id",
        StringType(),
        False
    ),
    StructField(
        "relationship_id",
        StringType(),
        False
    ),
    StructField(
        "relationship_name",
        StringType(),
        False
    ),
    StructField(
        "child_table",
        StringType(),
        False
    ),
    StructField(
        "child_columns",
        StringType(),
        False
    ),
    StructField(
        "parent_table",
        StringType(),
        False
    ),
    StructField(
        "parent_columns",
        StringType(),
        False
    ),
    StructField(
        "severity",
        StringType(),
        False
    ),
    StructField(
        "child_row_count",
        LongType(),
        False
    ),
    StructField(
        "evaluated_child_row_count",
        LongType(),
        False
    ),
    StructField(
        "child_missing_key_count",
        LongType(),
        False
    ),
    StructField(
        "matched_row_count",
        LongType(),
        False
    ),
    StructField(
        "orphan_row_count",
        LongType(),
        False
    ),
    StructField(
        "orphan_percentage",
        DoubleType(),
        False
    ),
    StructField(
        "status",
        StringType(),
        False
    )
])

if not relationship_summary:
    raise ValueError(
        "No relationship profiling results were generated."
    )

relationship_summary_df = spark.createDataFrame(
    relationship_summary,
    schema=relationship_profile_schema
)

display(
    relationship_summary_df.orderBy(
        "relationship_id"
    )
)

(
    relationship_summary_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_relationship_summary")
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e75d38d-a579-4985-b3f4-77813b3fb08e)

# Dataset Health Overview

consolidate profiling metrics into a single dataset health summary that can later power the Data Quality dashboard.

In [62]:
table_summary = spark.table("profile_table_summary")
column_summary = spark.table("profile_column_summary")
null_summary = spark.table("profile_null_summary")
duplicate_summary = spark.table("profile_duplicate_summary")
key_summary = spark.table("profile_key_quality_summary")
business_summary = spark.table("profile_business_rule_summary")
relationship_summary = spark.table("profile_relationship_summary")

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 68, Finished, Available, Finished, False)

In [64]:
null_overview = (
    null_summary
    .groupBy("table_name")
    .agg(
        F.sum("null_count").alias(
            "total_null_values"
        ),
        F.sum(
            F.when(
                F.col("null_count") > 0,
                1
            ).otherwise(0)
        ).alias(
            "columns_with_nulls"
        )
    )
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 70, Finished, Available, Finished, False)

In [65]:
business_overview = (
    business_summary
    .groupBy("table_name")
    .agg(
        F.sum(
            "failed_row_count"
        ).alias(
            "business_rule_failures"
        )
    )
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 71, Finished, Available, Finished, False)

In [66]:
relationship_overview = (
    relationship_summary
    .groupBy(
        "child_table"
    )
    .agg(
        F.sum(
            "orphan_row_count"
        ).alias(
            "relationship_failures"
        )
    )
    .withColumnRenamed(
        "child_table",
        "table_name"
    )    
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 72, Finished, Available, Finished, False)

In [68]:
overview = (
    table_summary
    .join(
        null_overview,
        "table_name",
        "left"
    )
    .join(
        duplicate_summary.select(
            "table_name",
            "exact_duplicate_row_count"
        ),
        "table_name",
        "left"
    )
    .join(
        key_summary.select(
            "table_name",
            "rows_with_missing_key"
        ),
        "table_name",
        "left"
    )
    .join(
        business_overview,
        "table_name",
        "left"
    )
    .join(
        relationship_overview,
        "table_name",
        "left"
    )
)

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 74, Finished, Available, Finished, False)

In [70]:
overview = overview.fillna({
    "columns_with_nulls": 0,
    "total_null_values": 0,
    "exact_duplicate_row_count": 0,
    "rows_with_missing_key": 0,
    "business_rule_failures": 0,
    "relationship_failures": 0
})

overview = (
    overview
    .withColumn(
        "overall_status",
        F.when(
            (F.col("relationship_failures") > 0)
            |
            (F.col("business_rule_failures") > 0),
            "Critical"
        )
        .when(
            (F.col("columns_with_nulls") > 0)
            |
            (F.col("exact_duplicate_row_count") > 0),
            "Warning"
        )
        .otherwise(
            "Healthy"
        )
    )
)

overview = (
    overview
    .withColumn(
        "profile_run_id",
        F.lit(
            PROFILE_RUN_ID
        )
    )
    .withColumn(
        "profiled_at",
        F.current_timestamp()
    )
)

display(overview.orderBy("table_name"))

StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 76, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c049466-8975-47c1-8625-0987ae0c8992)

In [71]:
overview.write.mode("overwrite").format("delta").saveAsTable("profile_overview")


StatementMeta(, e1bce971-62ed-4097-a6b3-109fefc36ccd, 77, Finished, Available, Finished, False)